# TP2_suite : Optimizing Hyperparameters (Keras)
**ESI-SBA — École Supérieure en Informatique 08-Mai-1945**  
**Deep Learning — Dr. N. DIF**

---
**Dataset:** CIFAR-10 (60 000 images 32×32 RGB, 10 classes)  
**Architecture:** Same MLP as TP2 → Input(3072) → Dense(128,ReLU) → Dense(64,ReLU) → Dense(10,Softmax)

## 0. Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import random
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

print('TensorFlow version:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

---
## 1. Load and Preprocess CIFAR-10

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

# Normalize pixel values to [0, 1]
X_train_full = X_train_full.astype('float32') / 255.0
X_test       = X_test.astype('float32') / 255.0

# Flatten 32x32x3 → 3072
X_train_full = X_train_full.reshape(-1, 3072)
X_test       = X_test.reshape(-1, 3072)

# One-hot encode labels
y_train_full = to_categorical(y_train_full, 10)
y_test       = to_categorical(y_test, 10)

print(f'Full train set : {X_train_full.shape}  Labels: {y_train_full.shape}')
print(f'Test set       : {X_test.shape}  Labels: {y_test.shape}')

In [ ]:
# Visualize sample CIFAR-10 images
(X_vis, y_vis), _ = cifar10.load_data()
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_vis[i])
    ax.set_title(CLASS_NAMES[y_vis[i][0]], fontsize=7)
    ax.axis('off')
plt.suptitle('Sample CIFAR-10 images (original)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. MLP Architecture (same as TP2)

| Layer    | Units | Activation | Parameters |
|----------|-------|------------|------------|
| Input    | 3072  | —          | —          |
| Dense 1  | 128   | ReLU       | 393,344    |
| Dense 2  | 64    | ReLU       | 8,256      |
| Output   | 10    | Softmax    | 650        |
| **Total**|       |            | **402,250**|

In [ ]:
def build_mlp(l2_reg=None, dropout_rate=None, batch_norm=False):
    """Build the MLP model with optional L2, Dropout, BatchNorm."""
    model = Sequential()
    model.add(Input(shape=(3072,)))
    
    # Hidden layer 1
    model.add(Dense(128, activation='relu'))
    if batch_norm:
        model.add(BatchNormalization())
    
    # Hidden layer 2 — with optional L2 and Dropout
    if l2_reg:
        model.add(Dense(64, activation='relu', kernel_regularizer=l2(l2_reg)))
    else:
        model.add(Dense(64, activation='relu'))
    
    if dropout_rate:
        model.add(Dropout(dropout_rate))
    
    # Output layer
    model.add(Dense(10, activation='softmax'))
    return model

# Display base architecture
demo = build_mlp()
demo.summary()

---
## 3. Train / Validation Split (80% / 20%)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, random_state=42, stratify=y_train_full
)

print(f'Train      : {X_train.shape}')
print(f'Validation : {X_val.shape}')
print(f'Test       : {X_test.shape}')

---
## 4. Baseline: Mini-Batch SGD (bs=128, lr=0.01, 50 epochs)

In [ ]:
print('Training Baseline MLP — Mini-Batch SGD (bs=128, lr=0.01, 50 epochs)...')

model_base = build_mlp()
model_base.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

t0 = time.time()
history_base = model_base.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=128, verbose=1
)
time_base = time.time() - t0

test_loss_base, test_acc_base = model_base.evaluate(X_test, y_test, verbose=0)
print(f'\n✅ Baseline done — {time_base:.1f}s')
print(f'   Train Acc : {history_base.history["accuracy"][-1]:.4f}')
print(f'   Val Acc   : {history_base.history["val_accuracy"][-1]:.4f}')
print(f'   Test Acc  : {test_acc_base:.4f}')

In [ ]:
def plot_curves(history, title, color='steelblue'):
    """Plot loss and accuracy learning curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    
    ax1.plot(history.history['loss'],     color=color, linewidth=2,   label='Train Loss')
    ax1.plot(history.history['val_loss'], color=color, linewidth=2,
             linestyle='--', alpha=0.7, label='Val Loss')
    ax1.set_title(f'{title} — Loss', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(); ax1.grid(alpha=0.3)
    
    ax2.plot(history.history['accuracy'],     color=color, linewidth=2,   label='Train Acc')
    ax2.plot(history.history['val_accuracy'], color=color, linewidth=2,
             linestyle='--', alpha=0.7, label='Val Acc')
    ax2.set_title(f'{title} — Accuracy', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.legend(); ax2.grid(alpha=0.3)
    
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_curves(history_base, 'Section 4 — Baseline: Mini-Batch SGD (bs=128, lr=0.01)')

### 📝 Remarks — Section 4
- CIFAR-10 is significantly harder than MNIST (RGB images, 10 varied classes).
- A flat MLP with only 2 hidden layers struggles: expected val accuracy ~40-45%.
- Clear overfitting visible: train accuracy keeps rising while val accuracy plateaus.
- This baseline will serve as reference for all next sections.

---
## 5. L2 Regularization on Dense Layer 2

In [ ]:
print('Training MLP with L2 regularization (lambda=0.001)...')

model_l2 = build_mlp(l2_reg=0.001)
model_l2.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

t0 = time.time()
history_l2 = model_l2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=128, verbose=1
)
time_l2 = time.time() - t0

test_loss_l2, test_acc_l2 = model_l2.evaluate(X_test, y_test, verbose=0)
print(f'\n✅ L2 Regularization done — {time_l2:.1f}s')
print(f'   Train Acc : {history_l2.history["accuracy"][-1]:.4f}')
print(f'   Val Acc   : {history_l2.history["val_accuracy"][-1]:.4f}')
print(f'   Test Acc  : {test_acc_l2:.4f}')

In [ ]:
plot_curves(history_l2, 'Section 5 — MLP with L2 Regularization (λ=0.001)', color='#e74c3c')

In [ ]:
# Comparison: Baseline vs L2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

epochs = range(1, 51)

ax1.plot(epochs, history_base.history['val_loss'], color='steelblue', linewidth=2, label='Baseline (no reg)')
ax1.plot(epochs, history_l2.history['val_loss'],   color='#e74c3c',   linewidth=2, label='With L2 (λ=0.001)')
ax1.set_title('Validation Loss: Baseline vs L2', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Val Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history_base.history['val_accuracy'], color='steelblue', linewidth=2, label='Baseline (no reg)')
ax2.plot(epochs, history_l2.history['val_accuracy'],   color='#e74c3c',   linewidth=2, label='With L2 (λ=0.001)')
ax2.set_title('Validation Accuracy: Baseline vs L2', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Section 5 — Comparison: Baseline vs L2 Regularization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary
print('\n── Section 5 Summary ─────────────────────────────────────')
print(f'{"Model":<25} {"Train Acc":>10} {"Val Acc":>10} {"Train-Val Gap":>14}')
print('-' * 62)
for name, hist in [("Baseline (no L2)", history_base), ("With L2 (λ=0.001)", history_l2)]:
    ta = hist.history['accuracy'][-1]
    va = hist.history['val_accuracy'][-1]
    print(f'{name:<25} {ta:>10.4f} {va:>10.4f} {ta-va:>14.4f}')

### 📝 Remarks — Section 5
- L2 regularization penalizes large weights, reducing overfitting.
- The train-val accuracy gap should decrease with L2.
- Train accuracy may be slightly lower, but val accuracy should improve.
- λ=0.001 is a typical starting value — too high → underfitting, too low → no effect.

---
## 6. Dropout Regularization (rates: 0.2, 0.3, 0.5)

In [ ]:
dropout_rates = [0.2, 0.3, 0.5]
histories_dropout = {}
times_dropout = {}
test_accs_dropout = {}

for rate in dropout_rates:
    print(f'Training with Dropout(rate={rate})...')
    model_dp = build_mlp(dropout_rate=rate)
    model_dp.compile(
        optimizer=SGD(learning_rate=0.01),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    t0 = time.time()
    histories_dropout[rate] = model_dp.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50, batch_size=128, verbose=0
    )
    times_dropout[rate] = time.time() - t0
    _, test_accs_dropout[rate] = model_dp.evaluate(X_test, y_test, verbose=0)
    va = histories_dropout[rate].history['val_accuracy'][-1]
    print(f'  → Val Acc: {va:.4f}  Test Acc: {test_accs_dropout[rate]:.4f}  Time: {times_dropout[rate]:.1f}s')

In [ ]:
# Individual curves for each dropout rate
colors_dp = ['#2980b9', '#27ae60', '#e74c3c']
fig, axes = plt.subplots(3, 2, figsize=(13, 13))

for i, (rate, col) in enumerate(zip(dropout_rates, colors_dp)):
    hist = histories_dropout[rate]
    axes[i,0].plot(hist.history['loss'],     color=col, linewidth=2, label='Train Loss')
    axes[i,0].plot(hist.history['val_loss'], color=col, linewidth=2, linestyle='--', label='Val Loss')
    axes[i,0].set_title(f'Dropout={rate} — Loss', fontweight='bold')
    axes[i,0].set_xlabel('Epoch'); axes[i,0].set_ylabel('Loss')
    axes[i,0].legend(); axes[i,0].grid(alpha=0.3)
    
    axes[i,1].plot(hist.history['accuracy'],     color=col, linewidth=2, label='Train Acc')
    axes[i,1].plot(hist.history['val_accuracy'], color=col, linewidth=2, linestyle='--', label='Val Acc')
    axes[i,1].set_title(f'Dropout={rate} — Accuracy', fontweight='bold')
    axes[i,1].set_xlabel('Epoch'); axes[i,1].set_ylabel('Accuracy')
    axes[i,1].legend(); axes[i,1].grid(alpha=0.3)

plt.suptitle('Section 6 — Dropout Regularization: Loss & Accuracy Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Overlay comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(history_base.history['val_loss'],     color='black',    linewidth=2, linestyle=':', label='Baseline')
ax2.plot(history_base.history['val_accuracy'], color='black',    linewidth=2, linestyle=':', label='Baseline')

for rate, col in zip(dropout_rates, colors_dp):
    hist = histories_dropout[rate]
    ax1.plot(hist.history['val_loss'],     color=col, linewidth=2, label=f'Dropout={rate}')
    ax2.plot(hist.history['val_accuracy'], color=col, linewidth=2, label=f'Dropout={rate}')

ax1.set_title('Validation Loss — Dropout Comparison', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Val Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.set_title('Validation Accuracy — Dropout Comparison', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Section 6 — Overlay: Baseline vs Dropout Rates', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
print('\n── Section 6 Summary ─────────────────────────────────────────────')
print(f'{"Model":<20} {"Train Acc":>10} {"Val Acc":>10} {"Test Acc":>10} {"Gap":>8} {"Time":>8}')
print('-' * 70)
ta_b = history_base.history['accuracy'][-1]
va_b = history_base.history['val_accuracy'][-1]
print(f'{"Baseline":<20} {ta_b:>10.4f} {va_b:>10.4f} {test_acc_base:>10.4f} {ta_b-va_b:>8.4f} {time_base:>7.1f}s')
for rate in dropout_rates:
    hist = histories_dropout[rate]
    ta = hist.history['accuracy'][-1]
    va = hist.history['val_accuracy'][-1]
    print(f'{f"Dropout={rate}":<20} {ta:>10.4f} {va:>10.4f} {test_accs_dropout[rate]:>10.4f} {ta-va:>8.4f} {times_dropout[rate]:>7.1f}s')

### 📝 Remarks — Section 6
- Dropout randomly disables neurons during training, forcing the network to learn redundant representations.
- **Dropout=0.2**: Light regularization, small reduction in overfitting.
- **Dropout=0.3**: Good balance between regularization strength and accuracy.
- **Dropout=0.5**: Strong regularization — train accuracy drops significantly, val may improve or stabilize.
- Higher dropout reduces the train-val gap but can hurt final performance if too aggressive.

---
## 7. Early Stopping

In [ ]:
print('Training with Early Stopping (patience=10, monitor=val_loss)...')

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

model_es = build_mlp()
model_es.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

t0 = time.time()
history_es = model_es.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,           # High max — early stopping will interrupt
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)
time_es = time.time() - t0

stopped_epoch = early_stop.stopped_epoch
best_epoch    = stopped_epoch - early_stop.patience + 1

test_loss_es, test_acc_es = model_es.evaluate(X_test, y_test, verbose=0)
print(f'\n✅ Early Stopping done — {time_es:.1f}s')
print(f'   Training stopped at epoch : {stopped_epoch + 1}')
print(f'   Best epoch (restored)     : {best_epoch}')
print(f'   Val Acc at stop           : {history_es.history["val_accuracy"][-1]:.4f}')
print(f'   Test Acc                  : {test_acc_es:.4f}')

In [ ]:
actual_epochs = len(history_es.history['loss'])
ep_range = range(1, actual_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(ep_range, history_es.history['loss'],     color='#8e44ad', linewidth=2, label='Train Loss')
ax1.plot(ep_range, history_es.history['val_loss'], color='#8e44ad', linewidth=2, linestyle='--', label='Val Loss')
ax1.axvline(x=best_epoch, color='red', linestyle=':', linewidth=2, label=f'Best epoch ({best_epoch})')
ax1.set_title('Early Stopping — Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep_range, history_es.history['accuracy'],     color='#8e44ad', linewidth=2, label='Train Acc')
ax2.plot(ep_range, history_es.history['val_accuracy'], color='#8e44ad', linewidth=2, linestyle='--', label='Val Acc')
ax2.axvline(x=best_epoch, color='red', linestyle=':', linewidth=2, label=f'Best epoch ({best_epoch})')
ax2.set_title('Early Stopping — Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle(f'Section 7 — Early Stopping (stopped at epoch {stopped_epoch+1}, best={best_epoch})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n📌 Training stopped at epoch {stopped_epoch+1} / 200')
print(f'   Best weights restored from epoch {best_epoch}')
print(f'   Saved {200 - (stopped_epoch+1)} unnecessary epochs')

### 📝 Remarks — Section 7
- Early stopping monitors val_loss and stops when it stops improving.
- `patience=10` means training stops after 10 consecutive epochs with no improvement.
- `restore_best_weights=True` ensures we get the model from the best epoch, not the last.
- This avoids wasting time on epochs that only increase overfitting.

---
## 8. Batch Normalization after First Hidden Layer

In [ ]:
print('Training with Batch Normalization after Dense(128)...')

model_bn = build_mlp(batch_norm=True)
model_bn.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_bn.summary()

t0 = time.time()
history_bn = model_bn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=128, verbose=1
)
time_bn = time.time() - t0

test_loss_bn, test_acc_bn = model_bn.evaluate(X_test, y_test, verbose=0)
print(f'\n✅ Batch Normalization done — {time_bn:.1f}s')
print(f'   Train Acc : {history_bn.history["accuracy"][-1]:.4f}')
print(f'   Val Acc   : {history_bn.history["val_accuracy"][-1]:.4f}')
print(f'   Test Acc  : {test_acc_bn:.4f}')

In [ ]:
plot_curves(history_bn, 'Section 8 — Batch Normalization after Dense(128)', color='#16a085')

# Comparison: Baseline vs BatchNorm
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

epochs = range(1, 51)
ax1.plot(epochs, history_base.history['val_loss'], color='steelblue', linewidth=2, label='Baseline')
ax1.plot(epochs, history_bn.history['val_loss'],   color='#16a085',   linewidth=2, label='With BatchNorm')
ax1.set_title('Validation Loss: Baseline vs BatchNorm', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Val Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history_base.history['val_accuracy'], color='steelblue', linewidth=2, label='Baseline')
ax2.plot(epochs, history_bn.history['val_accuracy'],   color='#16a085',   linewidth=2, label='With BatchNorm')
ax2.set_title('Validation Accuracy: Baseline vs BatchNorm', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Section 8 — Baseline vs Batch Normalization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n── Section 8 Summary ────────────────────────────────────')
print(f'{"Model":<25} {"Train Acc":>10} {"Val Acc":>10} {"Test Acc":>10}')
print('-' * 58)
for name, hist, tacc in [("Baseline", history_base, test_acc_base),
                          ("With BatchNorm", history_bn, test_acc_bn)]:
    ta = hist.history['accuracy'][-1]
    va = hist.history['val_accuracy'][-1]
    print(f'{name:<25} {ta:>10.4f} {va:>10.4f} {tacc:>10.4f}')

### 📝 Remarks — Section 8
- Batch Normalization normalizes the activations of each layer, reducing internal covariate shift.
- It usually allows higher learning rates and speeds up convergence.
- On CIFAR-10 with an MLP, BN should improve convergence speed notably.
- It also has a slight regularization effect, reducing overfitting.

---
## 9. Random Search for Best Hyperparameters

Search space:
- `learning_rate` ∈ {0.001, 0.005, 0.01, 0.05, 0.1}
- `dropout_rate`  ∈ {0.0, 0.1, 0.2, 0.3, 0.4, 0.5}
- `batch_size`    ∈ {32, 64, 128, 256}

In [ ]:
# Random Search configuration
PARAM_SPACE = {
    'learning_rate': [0.001, 0.005, 0.01, 0.05, 0.1],
    'dropout_rate':  [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    'batch_size':    [32, 64, 128, 256]
}

N_TRIALS    = 15     # Number of random combinations to try
EPOCHS_RS   = 20     # Fewer epochs per trial for speed

random.seed(42)
np.random.seed(42)

rs_results = []

print(f'Running Random Search — {N_TRIALS} trials × {EPOCHS_RS} epochs\n')
print(f'{"Trial":<6} {"LR":<8} {"Dropout":<10} {"BS":<6} {"Val Acc":>9} {"Time":>8}')
print('-' * 52)

for trial in range(1, N_TRIALS + 1):
    lr   = random.choice(PARAM_SPACE['learning_rate'])
    dr   = random.choice(PARAM_SPACE['dropout_rate'])
    bs   = random.choice(PARAM_SPACE['batch_size'])
    
    model_rs = build_mlp(dropout_rate=dr if dr > 0 else None)
    model_rs.compile(
        optimizer=SGD(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    t0 = time.time()
    hist_rs = model_rs.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_RS, batch_size=bs, verbose=0
    )
    elapsed = time.time() - t0
    va = hist_rs.history['val_accuracy'][-1]
    
    rs_results.append({
        'trial': trial, 'lr': lr, 'dropout': dr, 'batch_size': bs,
        'val_acc': va, 'time': elapsed, 'history': hist_rs
    })
    print(f'{trial:<6} {lr:<8} {dr:<10} {bs:<6} {va:>9.4f} {elapsed:>7.1f}s')

# Find best trial
best = max(rs_results, key=lambda x: x['val_acc'])
print(f'\n🏆 Best Trial #{best["trial"]}: lr={best["lr"]}, dropout={best["dropout"]}, bs={best["batch_size"]} → Val Acc={best["val_acc"]:.4f}')

In [ ]:
# Retrain best config for full 50 epochs
print(f'\nRetraining best config for 50 epochs...')
print(f'  lr={best["lr"]}, dropout={best["dropout"]}, batch_size={best["batch_size"]}')

dr_best = best['dropout'] if best['dropout'] > 0 else None
model_best = build_mlp(dropout_rate=dr_best)
model_best.compile(
    optimizer=SGD(learning_rate=best['lr']),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
t0 = time.time()
history_best = model_best.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=best['batch_size'], verbose=1
)
time_best = time.time() - t0

_, test_acc_best = model_best.evaluate(X_test, y_test, verbose=0)
print(f'\n✅ Best config (50 ep) done — {time_best:.1f}s')
print(f'   Val Acc  : {history_best.history["val_accuracy"][-1]:.4f}')
print(f'   Test Acc : {test_acc_best:.4f}')

In [ ]:
# Random Search Results Plot
trials     = [r['trial']    for r in rs_results]
val_accs   = [r['val_acc']  for r in rs_results]
lrs_used   = [r['lr']       for r in rs_results]
drops_used = [r['dropout']  for r in rs_results]
bs_used    = [r['batch_size'] for r in rs_results]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Val Acc per trial
bar_colors = ['#2ecc71' if r['trial']==best['trial'] else '#3498db' for r in rs_results]
axes[0,0].bar(trials, val_accs, color=bar_colors, edgecolor='black', alpha=0.85)
axes[0,0].axhline(best['val_acc'], color='red', linestyle='--', linewidth=1.5, label=f'Best={best["val_acc"]:.4f}')
axes[0,0].set_title('Val Accuracy per Trial', fontweight='bold')
axes[0,0].set_xlabel('Trial'); axes[0,0].set_ylabel('Val Accuracy')
axes[0,0].legend(); axes[0,0].grid(axis='y', alpha=0.3)

# Val Acc vs Learning Rate
axes[0,1].scatter(lrs_used, val_accs, s=80, c='#e74c3c', edgecolors='black', zorder=3)
axes[0,1].set_xscale('log')
axes[0,1].set_title('Val Accuracy vs Learning Rate', fontweight='bold')
axes[0,1].set_xlabel('Learning Rate (log scale)'); axes[0,1].set_ylabel('Val Accuracy')
axes[0,1].grid(alpha=0.3)

# Val Acc vs Dropout Rate
axes[1,0].scatter(drops_used, val_accs, s=80, c='#8e44ad', edgecolors='black', zorder=3)
axes[1,0].set_title('Val Accuracy vs Dropout Rate', fontweight='bold')
axes[1,0].set_xlabel('Dropout Rate'); axes[1,0].set_ylabel('Val Accuracy')
axes[1,0].grid(alpha=0.3)

# Val Acc vs Batch Size
axes[1,1].scatter(bs_used, val_accs, s=80, c='#e67e22', edgecolors='black', zorder=3)
axes[1,1].set_title('Val Accuracy vs Batch Size', fontweight='bold')
axes[1,1].set_xlabel('Batch Size'); axes[1,1].set_ylabel('Val Accuracy')
axes[1,1].grid(alpha=0.3)

plt.suptitle('Section 9 — Random Search Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Best config learning curves
plot_curves(history_best,
    f'Section 9 — Best Config (lr={best["lr"]}, dropout={best["dropout"]}, bs={best["batch_size"]})',
    color='#2ecc71')

# Top 5 results
top5 = sorted(rs_results, key=lambda x: x['val_acc'], reverse=True)[:5]
print('\n── Top 5 Random Search Results ──────────────────────────────────')
print(f'{"Rank":<6} {"LR":<8} {"Dropout":<10} {"BS":<6} {"Val Acc":>9}')
print('-' * 42)
for i, r in enumerate(top5, 1):
    print(f'{i:<6} {r["lr"]:<8} {r["dropout"]:<10} {r["batch_size"]:<6} {r["val_acc"]:>9.4f}')

### 📝 Remarks — Section 9
- Random Search efficiently explores the hyperparameter space without testing all combinations.
- With 15 trials of 20 epochs each, we cover enough combinations to identify good regions.
- Learning rate has the most impact on final accuracy.
- Batch size affects training speed and gradient noise level.
- The best found configuration is then retrained for 50 full epochs.

---
## 10. Global Summary — All Results

In [ ]:
# ── Global overlay: all validation accuracy curves ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Baseline vs L2 vs BatchNorm
axes[0].plot(history_base.history['val_accuracy'], color='steelblue', linewidth=2, label='Baseline')
axes[0].plot(history_l2.history['val_accuracy'],   color='#e74c3c',   linewidth=2, label='+ L2 (λ=0.001)')
axes[0].plot(history_bn.history['val_accuracy'],   color='#16a085',   linewidth=2, label='+ BatchNorm')
axes[0].set_title('Regularization Methods', fontweight='bold', fontsize=11)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Accuracy')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Panel 2: Dropout comparison
axes[1].plot(history_base.history['val_accuracy'], color='black',    linewidth=2, linestyle=':', label='Baseline')
for rate, col in zip(dropout_rates, colors_dp):
    axes[1].plot(histories_dropout[rate].history['val_accuracy'], color=col, linewidth=2, label=f'Dropout={rate}')
axes[1].set_title('Dropout Rates', fontweight='bold', fontsize=11)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Accuracy')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Panel 3: ES + Best config
ep_es = range(1, len(history_es.history['val_accuracy'])+1)
axes[2].plot(history_base.history['val_accuracy'], color='steelblue', linewidth=2, label='Baseline')
axes[2].plot(ep_es, history_es.history['val_accuracy'], color='#8e44ad', linewidth=2, label='Early Stopping')
axes[2].plot(history_best.history['val_accuracy'], color='#2ecc71', linewidth=2, label='Random Search Best')
axes[2].set_title('Early Stopping & Best Config', fontweight='bold', fontsize=11)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Val Accuracy')
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.suptitle('Section 10 — Global Comparison: Validation Accuracy (All Experiments)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Final bar chart: Test Accuracy all configs ─────────────────────
all_names = [
    'Baseline', 'L2 (λ=0.001)', 'Dropout=0.2', 'Dropout=0.3',
    'Dropout=0.5', 'Early Stop', 'BatchNorm', 'RS Best'
]
all_val_accs = [
    history_base.history['val_accuracy'][-1],
    history_l2.history['val_accuracy'][-1],
    histories_dropout[0.2].history['val_accuracy'][-1],
    histories_dropout[0.3].history['val_accuracy'][-1],
    histories_dropout[0.5].history['val_accuracy'][-1],
    history_es.history['val_accuracy'][-1],
    history_bn.history['val_accuracy'][-1],
    history_best.history['val_accuracy'][-1]
]
all_test_accs = [
    test_acc_base, test_acc_l2,
    test_accs_dropout[0.2], test_accs_dropout[0.3], test_accs_dropout[0.5],
    test_acc_es, test_acc_bn, test_acc_best
]
bar_colors_all = ['#3498db','#e74c3c','#2980b9','#27ae60','#c0392b','#8e44ad','#16a085','#2ecc71']

x = np.arange(len(all_names)); w = 0.35
fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x - w/2, [v*100 for v in all_val_accs],  w, label='Val Acc',  color=bar_colors_all, alpha=0.6, edgecolor='white')
b2 = ax.bar(x + w/2, [v*100 for v in all_test_accs], w, label='Test Acc', color=bar_colors_all, alpha=0.95, edgecolor='black', linewidth=0.5)
for b, v in zip(b2, [v*100 for v in all_test_accs]):
    ax.text(b.get_x()+b.get_width()/2, v+0.2, f'{v:.1f}', ha='center', fontsize=7.5, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Final Results — Val & Test Accuracy: All Configurations', fontweight='bold', fontsize=12)
ax.set_xticks(x); ax.set_xticklabels(all_names, fontsize=9)
ax.set_ylim(0, 75)
ax.legend(['Val Acc (light)', 'Test Acc (dark)'], fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Overfitting gap comparison ──────────────────────────────────────
all_train_accs = [
    history_base.history['accuracy'][-1],
    history_l2.history['accuracy'][-1],
    histories_dropout[0.2].history['accuracy'][-1],
    histories_dropout[0.3].history['accuracy'][-1],
    histories_dropout[0.5].history['accuracy'][-1],
    history_es.history['accuracy'][-1],
    history_bn.history['accuracy'][-1],
    history_best.history['accuracy'][-1]
]
gaps = [t - v for t, v in zip(all_train_accs, all_val_accs)]

fig, ax = plt.subplots(figsize=(12, 4))
bar_gap = ax.bar(all_names, [g*100 for g in gaps], color=bar_colors_all, edgecolor='black', alpha=0.85)
for b, g in zip(bar_gap, [g*100 for g in gaps]):
    ax.text(b.get_x()+b.get_width()/2, g+0.1, f'{g:.1f}%', ha='center', fontsize=8, fontweight='bold')
ax.set_ylabel('Train Acc − Val Acc (%)', fontsize=11)
ax.set_title('Overfitting Gap: All Configurations (lower = better generalization)', fontweight='bold', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Final summary table ─────────────────────────────────────────────
print('\n══════════════════════════════════════════════════════════════════')
print('                   GLOBAL SUMMARY TABLE')
print('══════════════════════════════════════════════════════════════════')
print(f'{"Config":<22} {"Train Acc":>10} {"Val Acc":>10} {"Test Acc":>10} {"Gap":>8}')
print('─' * 65)
for name, ta, va, testa in zip(all_names, all_train_accs, all_val_accs, all_test_accs):
    marker = ' ◄ BEST' if name == 'RS Best' else ''
    print(f'{name:<22} {ta:>10.4f} {va:>10.4f} {testa:>10.4f} {ta-va:>8.4f}{marker}')
print('═' * 65)
best_idx = np.argmax(all_test_accs)
print(f'\n🏆 Best Test Accuracy: {all_names[best_idx]} → {all_test_accs[best_idx]*100:.2f}%')
min_gap_idx = np.argmin(gaps)
print(f'📉 Smallest Overfit Gap: {all_names[min_gap_idx]} → {gaps[min_gap_idx]*100:.2f}%')

---
## General Conclusions

| # | Finding | Explanation |
|---|---------|-------------|
| 1 | **MLP struggles on CIFAR-10** | CIFAR-10 has complex spatial features. A flat MLP ignores spatial structure → ~40-50% accuracy vs ~99% for CNN. |
| 2 | **L2 reduces overfitting** | Penalizing large weights shrinks the train-val gap. Marginal accuracy gain but better generalization. |
| 3 | **Dropout=0.3 is the sweet spot** | Too low (0.2) = minor effect. Too high (0.5) = underfitting. 0.3 balances regularization and capacity. |
| 4 | **Early Stopping saves time** | Automatically stops before overfitting worsens, restoring the best weights. |
| 5 | **BatchNorm speeds up convergence** | Normalizing activations reduces internal covariate shift → faster, more stable training. |
| 6 | **Random Search finds better configs** | Efficiently explores the hyperparameter space without exhaustive grid search. |
| 7 | **CNN needed for CIFAR-10 > 80%** | Conv2D layers capture spatial features that Dense layers fundamentally cannot. |

---
*Notebook: TP2_suite — CIFAR-10 Hyperparameter Optimization | ESI-SBA Deep Learning*